## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [11]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 167.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 207.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 264.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 203.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.26.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.40.0 which is incompatible.
google-adk 1.26.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
langgraph-prebuilt 1.0.8 requires langchain-core>=1.0.0, but you have langchain-c

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [12]:
# For installing the libraries & downloading models from HF Hub
!pip install huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1 sentence-transformers==5.1.0 numpy==2.3.3 -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.3.3 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.3.3 which is incompatible.


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [140]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

#Pretty print helper
import pprint

def output(text):
  pprint.pprint(text, width=120)

## Question Answering using LLM

#### Downloading and Loading the model

In [14]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

llm = Llama(
    model_path=model_path,
    n_ctx=5000,
    n_gpu_layers=38,
    n_batch=512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


In [156]:
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    output( ">>>>>>>>>Calling llm with mt=" + str(max_tokens) + " temp=" + str(temperature) + " tp=" + str(top_p) + " tk=" + str(top_k) )
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [157]:
q1 = "What is the protocol for managing sepsis in a critical care unit?"
response(q1,max_tokens=128,temperature=0,top_p=0.95,top_k=50)


'>>>>>>>>>Calling llm with mt=128 temp=0 tp=0.95 tk=50'


Llama.generate: prefix-match hit


'\n\nSepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:\n\n1. Early recognition: Recognize the signs and symptoms of sepsis early and initiate treatment as soon as possible. Sepsis can present with various clinical features, including fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.\n2. ABCs'

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [143]:
q2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(q2,max_tokens=128,temperature=0,top_p=0.95,top_k=50)

Llama.generate: prefix-match hit


'\n\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small pouch-like structure that extends from the large intestine. The symptoms of appendicitis can vary from person to person, but some common signs include:\n\n1. Abdominal pain: The pain is typically located in the lower right side of the abdomen and may be constant or come and go. It may start as a mild discomfort that worsens over time.\n2. Loss of appetite: People with appendicitis often lose their appetite due to abdominal pain and nausea'

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [18]:
q3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(q3,max_tokens=128,temperature=0,top_p=0.95,top_k=50)

Llama.generate: prefix-match hit


"\n\nSudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles. It can result in round or oval bald patches on the scalp, but it can also occur on other parts of the body such as the beard area, eyebrows, and eyelashes.\n\nThe exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Some possible triggers for this condition include stress, genetics, viral infections, and certain medications."

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [19]:
q4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(q4,max_tokens=128,temperature=0,top_p=0.95,top_k=50)

Llama.generate: prefix-match hit


'\n\nA person who has sustained a physical injury to the brain tissue may require various treatments depending on the severity and location of the injury. Here are some common treatments that may be recommended:\n\n1. Emergency care: In case of a traumatic brain injury (TBI), it is essential to seek emergency medical attention as soon as possible. The primary goal of emergency care is to prevent further damage to the brain, stabilize vital signs, and manage any life-threatening conditions.\n2. Medications: Depending on the symptoms, healthcare professionals may prescribe medications to manage various conditions associated with a'

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [20]:
q5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(q5,max_tokens=128,temperature=0,top_p=0.95,top_k=50)

Llama.generate: prefix-match hit


"\n\nFirst and foremost, if you suspect that someone has fractured their leg while hiking, it's essential to ensure their safety and prevent further injury. Here are some necessary precautions:\n\n1. Keep the person calm and still: Encourage them to remain as still as possible to minimize pain and prevent worsening the injury.\n2. Assess the situation: Check for any signs of shock, such as pale skin, rapid heartbeat, or shallow breathing. If you notice these symptoms, seek medical help immediately.\n3. Immobilize the leg: Use a splint, sl"

**Observations:**


* When prompting the LLM without any context and using the default parameters (max_tokens, temperature, top_p, top_k) the responses are very limited in that they are vague, short, and sometimes truncated.
* The response time is very quick, less than 10 seconds.

## Question Answering using LLM with Prompt Engineering

In [21]:
llm_query_context = """
You are an expert whose work is to review the report and provide the appropriate answers from the context.
If the answer is not found in the context, respond "I don't know".
"""

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [160]:
#Five different LLM parameter combinations used in this notebook
paramList = data = [
    {"max_tokens": 1000, "temp": 0, "top_p" : 0.95, "top_k":  50},
    {"max_tokens": 1000, "temp": 1, "top_p" : 0.85, "top_k":  100},
    {"max_tokens": 1000, "temp": 2, "top_p" : 0.75, "top_k":  250},
    {"max_tokens": 1000, "temp": 3, "top_p" : 0.65, "top_k":  500},
    {"max_tokens": 1000, "temp": 4, "top_p" : 0.45, "top_k":  500},
]

In [162]:
for params in paramList:
  output(response(llm_query_context + q1,max_tokens=params["max_tokens"],temperature=params["temp"],top_p=params["top_p"],top_k=params["top_k"]))
  output("                                      ---------------------------")

'>>>>>>>>>Calling llm with mt=1000 temp=0 tp=0.95 tk=50'


Llama.generate: prefix-match hit


('\n'
 'Based on the context provided, the protocol for managing sepsis in a critical care unit may include the following '
 'steps:\n'
 '1. Early recognition and diagnosis of sepsis using clinical criteria such as the Sequential Organ Failure Assessment '
 '(SOFA) score or Quick Sequential Organ Failure Assessment (qSOFA) score.\n'
 '2. Immediate initiation of antibiotic therapy based on culture results, if available, or empiric therapy based on '
 "the patient's clinical presentation and local guidelines.\n"
 '3. Fluid resuscitation to maintain adequate tissue perfusion and organ function, with careful monitoring of fluid '
 'balance and hemodynamic status.\n'
 '4. Administration of vasopressors or inotropes as needed to maintain mean arterial pressure (MAP) above 65 mmHg.\n'
 '5. Close monitoring of vital signs, laboratory values, and organ function, including urine output, respiratory '
 'status, and cardiac output.\n'
 '6. Use of adjunctive therapies such as corticosteroids, vasoa

Llama.generate: prefix-match hit


('\n'
 'Based on the context of the following report, the protocol for managing sepsis in a critical care unit would '
 'involve:\n'
 '1. Early recognition and diagnosis through assessment of vital signs, laboratory results, and clinical '
 'presentation.\n'
 '2. Administration of broad-spectrum antibiotics as soon as possible.\n'
 '3. Fluid resuscitation to maintain adequate tissue perfusion.\n'
 '4. Close monitoring of hemodynamic status, organ function, and laboratory values.\n'
 '5. Appropriate supportive measures, such as mechanical ventilation or vasopressors, if needed.\n'
 '6. Source control, such as drainage of an abscess or debridement of necrotic tissue, to eliminate the infection '
 'source.\n'
 '7. Adjustment of treatment based on response to therapy and patient condition.\n'
 '8. Consideration of adjunctive therapies, such as corticosteroids or anticoagulation, if indicated by clinical '
 'guidelines.\n'
 'The report does not provide specific details on the protocol used 

...

('\n'
 'The management of sepsis in a critical care unit involves a systematic approach that focuses on early recognition, '
 'rapid response, and effective treatment. The following are key components of sepsis management in a critical care '
 'setting:\n'
 '1. Recognition: Early identification of sepsis is crucial for timely intervention. Assess patients for signs and '
 'symptoms of infection, organ dysfunction, and septic shock. Utilize clinical scoring systems such as the Sequential '
 'Organ Failure Assessment (SOFA) score or the Quick Sequential Organ Failure Assessment (qSOFA) score to help '
 'identify patients at risk for sepsis.\n'
 '2. Resuscitation: If sepsis is suspected, initiate resuscitation measures promptly. This includes administering '
 'oxygen and intravenous fluids to maintain adequate tissue perfusion. Use vasopressors if needed to maintain mean '
 'arterial pressure (MAP) ≥65 mmHg and avoid hypotension.\n'
 '3. Antimicrobial therapy: Start appropriate antimicrob

Llama.generate: prefix-match hit


('\n'
 'The protocol for managing sepsis in a critical care unit typically involves the following steps:\n'
 '1. Early recognition and suspicion of sepsis based on clinical signs and symptoms such as fever, tachycardia, '
 'tachypnea, altered mental status, and lactic acidosis.\n'
 '2. Immediate initiation of antibiotic therapy based on culture results or suspected source of infection.\n'
 '3. Fluid resuscitation to maintain adequate tissue perfusion and organ function. This may involve the use of '
 "crystalloids, colloids, or blood products depending on the patient's response.\n"
 '4. Close monitoring of vital signs, laboratory values, and organ function.\n'
 '5. Adjustment of antibiotic therapy based on culture results and sensitivity data.\n'
 '6. Optimization of oxygenation and ventilation as needed.\n'
 '7. Use of vasopressors to maintain adequate blood pressure and tissue perfusion.\n'
 '8. Management of co-morbidities such as diabetes, renal dysfunction, or cardiac disease.\n'


Llama.generate: prefix-match hit


('\n'
 'According to the Surviving Sepsis Campaign guidelines, the following steps should be taken for managing sepsis in a '
 'critical care unit:\n'
 '1. Early recognition and diagnosis: Identify sepsis early and initiate treatment as soon as possible. Use the '
 'Sequential [Sepsis-related] Organ Failure Assessment (SOFA) score to assess severity of illness.\n'
 '2. Source control: Address the source of infection, if possible. This may involve surgical intervention or other '
 'interventions such as drainage or debridement.\n'
 '3. Fluid resuscitation: Administer fluids to maintain adequate tissue perfusion and organ function. Use crystalloids '
 'initially, followed by colloids if needed.\n'
 '4. Vasopressor therapy: Use vasopressors to maintain mean arterial pressure (MAP) ≥65 mmHg in patients with '
 'sepsis-induced hypotension.\n'
 '5. Steroid therapy: Consider using corticosteroids in patients with septic shock who have not responded to fluid '
 'resuscitation and vasopressor t

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [163]:
for params in paramList:
  output(response(llm_query_context + q2,max_tokens=params["max_tokens"],temperature=params["temp"],top_p=params["top_p"],top_k=params["top_k"]))
  output("                                      ---------------------------")

'>>>>>>>>>Calling llm with mt=1000 temp=0 tp=0.95 tk=50'


Llama.generate: prefix-match hit


('\n'
 'According to the Mayo Clinic, the common symptoms of appendicitis include:\n'
 '1. Sudden pain that starts near your navel and often shifts to your lower right abdomen.\n'
 '2. Loss of appetite.\n'
 '3. Nausea and vomiting.\n'
 '4. Abdominal swelling.\n'
 '5. Fever, which may be low-grade at first but can rise to 101 degrees Fahrenheit or higher as the illness '
 'progresses.\n'
 '6. Diarrhea or constipation.\n'
 '7. Feeling sick, lethargic or generally unwell.\n'
 '8. Abdominal pain that worsens if you cough, sneeze, or make other jarring movements.\n'
 '9. Inability to pass gas or have a bowel movement.\n'
 '10. Pain in the lower back on the right side.\n'
 'Appendicitis cannot be cured via medicine alone. If left untreated, it can lead to a rupture of the appendix, which '
 'can result in peritonitis, a serious inflammation of the abdominal cavity that requires immediate medical attention. '
 'The standard treatment for appendicitis is surgical removal of the appendix, known

Llama.generate: prefix-match hit


('\n'
 'According to the given context, the common symptoms for appendicitis include:\n'
 '1. Pain in the lower right abdomen, often starting around the navel and then shifting to the lower right side.\n'
 '2. Loss of appetite and feeling sick to your stomach.\n'
 '3. Fever and chills.\n'
 '4. Feeling bloated or swollen in the abdomen.\n'
 '5. Inability to pass gas or have a bowel movement.\n'
 'As for the treatment, the context states that appendicitis cannot be cured via medicine alone and usually requires '
 'surgery. The standard surgical procedure for treating appendicitis is called an appendectomy, which involves '
 'removing the appendix through an incision in the abdomen.')
'                                      ---------------------------'
'>>>>>>>>>Calling llm with mt=1000 temp=2 tp=0.75 tk=250'


...

('\n'
 'Based on the provided context, the common symptoms for appendicitis include abdominal pain, usually located in the '
 'lower right side of the abdomen, loss of appetite, nausea or vomiting, and fever. Appendicitis cannot be cured via '
 'medicine alone; surgery is necessary to remove the infected appendix. The surgical procedure commonly used to treat '
 'appendicitis is called an appendectomy. This involves making a small incision in the abdomen and removing the '
 'appendix. Sometimes, a laparoscopic appendectomy may be performed, which uses smaller incisions and a camera to '
 'guide the surgeon. However, the specific procedure used depends on the individual case and the judgement of the '
 'treating physician.')
'                                      ---------------------------'
'>>>>>>>>>Calling llm with mt=1000 temp=3 tp=0.65 tk=500'


Llama.generate: prefix-match hit


('\n'
 'According to the given context, the common symptoms for appendicitis include: abdominal pain, usually located in the '
 'lower right side of the abdomen; loss of appetite; nausea and vomiting; fever; and a feeling of not being able to '
 'pass gas or have a bowel movement.\n'
 'The context also states that appendicitis cannot be cured via medicine alone and requires surgical intervention for '
 'treatment. The most common surgical procedure used to treat appendicitis is an appendectomy, which involves removing '
 'the appendix through an incision in the abdomen.')
'                                      ---------------------------'
'>>>>>>>>>Calling llm with mt=1000 temp=4 tp=0.45 tk=500'


Llama.generate: prefix-match hit


('\n'
 'Based on the context provided, the common symptoms for appendicitis include abdominal pain, loss of appetite, '
 'nausea, vomiting, fever, and a feeling of illness in general. The context also states that appendicitis cannot be '
 'cured via medicine and requires surgical intervention. Therefore, the answer is:\n'
 'The common symptoms for appendicitis are abdominal pain, loss of appetite, nausea, vomiting, fever, and a feeling of '
 'illness in general. Appendicitis cannot be cured via medicine and requires surgical intervention. The appropriate '
 'surgical procedure to treat it is an appendectomy, which involves removing the inflamed appendix from the body.')
'                                      ---------------------------'


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [164]:
for params in paramList:
  output(response(llm_query_context + q3,max_tokens=params["max_tokens"],temperature=params["temp"],top_p=params["top_p"],top_k=params["top_k"]))
  output("                                      ---------------------------")

'>>>>>>>>>Calling llm with mt=1000 temp=0 tp=0.95 tk=50'


Llama.generate: prefix-match hit


('\n'
 'According to the American Academy of Dermatology (AAD), there are several potential causes for sudden, patchy hair '
 'loss, including:\n'
 '1. Alopecia Areata: This is an autoimmune disorder that causes hair loss in circular patches on the scalp, face, or '
 'other areas of the body. It can affect people of all ages and ethnicities. There is no cure for alopecia areata, but '
 'various treatments can help stimulate hair regrowth. These include:\n'
 '   a. Corticosteroids: Topical or injected steroids can reduce inflammation and promote hair growth.\n'
 '   b. Immunotherapy: Injections of certain substances can help the immune system to attack the cells that cause hair '
 'loss.\n'
 '   c. Minoxidil: A topical medication that can stimulate hair growth in some people.\n'
 '2. Telogen Effluvium: This is a condition where the hair enters the resting phase prematurely, leading to shedding. '
 'It can be caused by stress, illness, or certain medications. Treatment for telogen effluv

Llama.generate: prefix-match hit


('\n'
 'The context provided does not mention any specific treatments or solutions for sudden patchy hair loss. However, '
 'there are several possible causes that can lead to this condition, including:\n'
 '1. Alopecia Areata - an autoimmune disease that causes hair loss in small round patches on the scalp, beard, or '
 'other areas of the body.\n'
 '2. Trauma or injury to the scalp, such as burning, cutting, or pulling out hairs.\n'
 '3. Nutritional deficiencies, particularly iron, zinc, or vitamin B12.\n'
 '4. Hormonal imbalances, such as thyroid disorders or polycystic ovary syndrome (PCOS).\n'
 '5. Stress or emotional distress.\n'
 '6. Certain medications, such as chemotherapy drugs or certain blood thinners.\n'
 '7. Infections, such as ringworm or folliculitis.\n'
 '8. Genetics.\n'
 "To address sudden patchy hair loss, it's essential to identify the underlying cause first. Once the cause is "
 'identified, appropriate treatments can be considered. For example:\n'
 '1. Alopecia Ar

...

('\n'
 '\n'
 'From the context provided, there is no specific information about the effective treatments or solutions for '
 'addressing sudden patchy hair loss. However, there are several possible causes mentioned, including stress, '
 'autoimmune disorders such as alopecia areata, nutritional deficiencies, hormonal imbalances, and certain '
 'medications.\n'
 '\n'
 'For stress-induced hair loss, practicing relaxation techniques such as meditation, yoga, or deep breathing exercises '
 'may help reduce stress levels and promote hair growth. Additionally, a healthy diet rich in essential vitamins and '
 'minerals can support hair health and prevent nutritional deficiencies. For hormonal imbalances, treating the '
 'underlying condition may be necessary to restore hair growth.\n'
 '\n'
 'For autoimmune disorders such as alopecia areata, there is no cure, but various treatments can help promote hair '
 'regrowth or reduce inflammation. These include corticosteroids, immunosuppressants, an

Llama.generate: prefix-match hit


('\n'
 '\n'
 "From the context provided, there isn't enough information to determine the specific causes of sudden patchy hair "
 'loss or the most effective treatments. However, I can provide some general information based on common causes and '
 'treatments for this condition.\n'
 '\n'
 'Possible causes of sudden patchy hair loss include:\n'
 '\n'
 '1. Alopecia Areata: This is an autoimmune disease that attacks hair follicles, causing hair to fall out in small '
 'patches. It can affect anyone, regardless of age or gender.\n'
 '2. Stress: Sudden stress or trauma can cause temporary hair loss, known as telogen effluvium. This condition usually '
 'resolves on its own within a few months.\n'
 '3. Nutritional deficiencies: Lack of certain nutrients, such as iron, zinc, or biotin, can lead to hair loss.\n'
 '4. Hormonal imbalances: Hormonal changes, such as those that occur during pregnancy or menopause, can cause hair '
 'loss.\n'
 '5. Medications: Certain medications, including antidep

Llama.generate: prefix-match hit


('\n'
 'According to the American Academy of Dermatology (AAD), the most common cause of sudden patchy hair loss is a '
 'condition called alopecia areata. Alopecia areata is an autoimmune disease that attacks hair follicles, resulting in '
 'circular bald spots on the scalp or other areas of the body.\n'
 'Effective treatments for addressing sudden patchy hair loss due to alopecia areata include:\n'
 '1. Corticosteroids: These medications can be applied directly to the affected area or taken orally to reduce '
 'inflammation and suppress the immune system, allowing hair to grow back.\n'
 '2. Immunotherapy: This treatment involves injecting the affected area with a substance that stimulates the immune '
 'system to attack the cause of the hair loss rather than the hair follicles themselves.\n'
 '3. Minoxidil: This medication is applied topically and can help slow down hair loss and promote new growth in some '
 'cases.\n'
 '4. Hair transplantation: In severe cases, hair transplantation

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [165]:
for params in paramList:
  output(response(llm_query_context + q4,max_tokens=params["max_tokens"],temperature=params["temp"],top_p=params["top_p"],top_k=params["top_k"]))
  output("                                      ---------------------------")

'>>>>>>>>>Calling llm with mt=1000 temp=0 tp=0.95 tk=50'


Llama.generate: prefix-match hit


('\n'
 'Based on the context provided, there is no specific information about the recommended treatments for a person with a '
 'physical injury to brain tissue and resulting impairment of brain function. However, some common treatments for '
 'brain injuries include:\n'
 '1. Medications to manage symptoms such as pain, swelling, or seizures.\n'
 '2. Rehabilitation therapies such as physical therapy, occupational therapy, speech therapy, and cognitive '
 'rehabilitation.\n'
 '3. Surgery to remove hematomas or other lesions that may be causing pressure on the brain.\n'
 '4. Assistive devices such as braces, prosthetics, or communication aids to help with daily activities.\n'
 '5. Psychological counseling to address emotional and behavioral changes that may occur after a brain injury.\n'
 'It is important to note that the specific treatment plan will depend on the severity and location of the injury, as '
 "well as the individual's age, overall health, and personal needs. Therefore, it i

Llama.generate: prefix-match hit


('\n'
 'Based on the provided context, there is no explicit mention of specific treatments recommended for a person with a '
 'brain injury. However, some common therapies and interventions used to manage brain injuries include:\n'
 '- Rehabilitation therapy (physical therapy, occupational therapy, speech therapy) to help restore lost functions and '
 'improve overall quality of life.\n'
 '- Medications to manage symptoms such as pain, seizures, and depression.\n'
 '- Surgery in some cases, such as removing hematomas or repairing skull fractures.\n'
 '- Assistive devices, such as prosthetics or communication aids, to help individuals function more independently.\n'
 "It's important to note that every brain injury is unique, so the specific treatment plan will depend on the severity "
 "and location of the injury, as well as the individual's overall health and personal circumstances.\n"
 'Therefore, I would recommend consulting with a healthcare professional for an accurate diagnosis an

...

('\n'
 'The specific treatment recommendations for a person with a brain injury can depend on various factors such as the '
 "severity and location of the injury, the extent of the functional impairment, and the individual's overall health "
 'condition. However, some common treatments that may be recommended include:\n'
 '1. Medications to manage symptoms such as pain, inflammation, seizures, or depression.\n'
 '2. Rehabilitation therapies such as physical therapy, occupational therapy, speech therapy, and cognitive '
 'rehabilitation to help restore lost functions and improve overall functioning.\n'
 '3. Assistive devices such as wheelchairs, braces, communication aids, or adaptive equipment to help individuals '
 'function more independently.\n'
 '4. Surgery, if necessary, to remove hematomas or other lesions that may be compressing brain tissue or causing '
 'further damage.\n'
 '5. Nutritional support and proper diet to ensure adequate nutrition for brain healing and recovery.\n'


Llama.generate: prefix-match hit


('\n'
 'According to the Mayo Clinic, treatment for a brain injury depends on the severity and location of the injury. For '
 'mild injuries, such as concussions, rest, hydration, and avoiding activities that can worsen symptoms are '
 'recommended. More severe injuries may require hospitalization, surgery, rehabilitation therapy, or assistive devices '
 'to help with daily living activities. Common therapies include physical therapy, occupational therapy, '
 'speech-language therapy, and cognitive rehabilitation. Medications may also be prescribed to manage symptoms such as '
 'pain, swelling, or seizures. In some cases, ongoing care and support may be necessary for managing the long-term '
 'effects of a brain injury. (Source: Mayo Clinic)\n'
 'Therefore, the recommended treatments for a person who has sustained a physical injury to brain tissue, resulting in '
 'temporary or permanent impairment of brain function include rest, hydration, avoiding activities that can worsen '
 'sympt

Llama.generate: prefix-match hit


('\n'
 'The context does not provide specific information on the recommended treatments for a person with a physical injury '
 'to brain tissue resulting in temporary or permanent impairment of brain function. However, some common treatments '
 'may include:\n'
 '1. Medications to manage symptoms such as pain, seizures, or depression.\n'
 '2. Rehabilitation therapy, including physical therapy, occupational therapy, speech therapy, and cognitive '
 'rehabilitation.\n'
 '3. Surgery to remove hematomas or other lesions that may be causing pressure on the brain.\n'
 '4. Assistive devices such as braces, prosthetics, or communication aids.\n'
 '5. Lifestyle modifications, such as diet and exercise, to promote overall health and well-being.\n'
 'It is important to note that the specific treatment plan will depend on the severity and location of the injury, as '
 "well as the individual's age, overall health, and personal goals. Therefore, it is recommended that individuals "
 'consult with a

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [166]:
for params in paramList:
  output(response(llm_query_context + q5,max_tokens=params["max_tokens"],temperature=params["temp"],top_p=params["top_p"],top_k=params["top_k"]))
  output("                                      ---------------------------")

'>>>>>>>>>Calling llm with mt=1000 temp=0 tp=0.95 tk=50'


Llama.generate: prefix-match hit


('\n'
 'According to Mayo Clinic, if you suspect that you or someone else has fractured a leg during a hiking trip, follow '
 'these steps:\n'
 '1. Stay calm and try to keep the person as still as possible to prevent further injury.\n'
 '2. Apply a splint or immobilize the leg with a makeshift sling or other material if possible.\n'
 '3. Do not attempt to realign the bone or push it back into place yourself.\n'
 '4. Seek medical attention as soon as possible, even if the fracture is not severe.\n'
 '5. In the meantime, try to keep the person warm and comfortable, provide fluids to prevent dehydration, and monitor '
 'their vital signs.\n'
 '6. If the person is in significant pain or unable to walk, they may need to be carried out of the hiking area on a '
 'stretcher or other means.\n'
 "7. Once the person has received medical attention, follow the healthcare provider's instructions for care and "
 'recovery, which may include wearing a cast or brace, using crutches or a walker, and at

Llama.generate: prefix-match hit


('\n'
 "Based on the context provided, there isn't enough information to provide specific precautions and treatment steps "
 'for a person who has fractured their leg during a hiking trip. However, I can suggest some general guidelines based '
 'on common medical advice.\n'
 '1. Assess the severity of the injury: If the fracture is suspected, do not move the person unnecessarily to avoid '
 'causing further damage. Try to keep them as comfortable as possible and prevent any unnecessary discomfort or pain.\n'
 '2. Provide first aid: Apply a sterile dressing to the wound if there is one. Splint the leg to provide support and '
 'stability to the fracture. Use a splint, branch, or other available material to immobilize the leg above and below '
 'the injury.\n'
 "3. Seek medical attention: Fractures require professional medical care, so it's essential to seek help as soon as "
 'possible. Inform the hiking group or emergency services if needed.\n'
 '4. Consider transportation: Depending o

...

('\n'
 'According to Mayo Clinic, if you suspect that you or someone else has fractured a leg during a hiking trip, follow '
 'these steps:\n'
 '1. Stay calm and try to make the person as comfortable as possible.\n'
 '2. Assess the extent of the injury by checking for signs of swelling, bruising, deformity, numbness, or inability to '
 'move the leg.\n'
 '3. Do not attempt to realign or manipulate the bone, as this can cause further damage or harm.\n'
 '4. Use a splint, sling, or other immobilizing device to prevent movement and help support the injured leg.\n'
 '5. If possible, have someone stay with the person while you go for help. Otherwise, call for emergency medical '
 'assistance.\n'
 'Regarding treatment steps, once the person has been evaluated by a healthcare professional, they may be prescribed '
 'the following:\n'
 '1. Pain medication to manage discomfort and make it easier to move the leg as needed.\n'
 '2. Immobilization with a cast or brace to keep the bone in place and

Llama.generate: prefix-match hit


('\n'
 'According to the American Academy of Orthopaedic Surgeons (AAOS), a fractured leg, also known as a broken leg, '
 'requires immediate medical attention. The following are the necessary precautions and treatment steps for a person '
 'who has fractured their leg during a hiking trip:\n'
 '1. Immediate Care: Apply ice to reduce swelling and pain. Do not move the injured leg unless it is necessary to '
 'prevent further injury or to get the person to safety. Splint the leg, if possible, using materials available in the '
 'environment such as sticks, clothes, or a hiking pole.\n'
 '2. Transportation: Arrange for transportation to the nearest medical facility. If the person cannot walk, they may '
 'need to be carried out on a stretcher or in a makeshift stretcher made from hiking poles and other materials.\n'
 '3. Medical Treatment: Once at the hospital, the doctor will evaluate the fracture and determine the best course of '
 'treatment, which may include setting the bone with pi

Llama.generate: prefix-match hit


('\n'
 'Based on the context provided, I cannot directly answer the question as there is no specific report or context given '
 'regarding a particular incident of a leg fracture during a hiking trip. However, I can provide some general '
 'information about necessary precautions and treatment steps for a person who has fractured their leg, as well as '
 'considerations for their care and recovery.\n'
 '1. Precautions:\n'
 '   - Do not move the person excessively or apply weight to the injured leg.\n'
 '   - Keep the leg stable and immobilized using a splint, sling, or brace.\n'
 '   - Avoid putting any pressure on the fracture site.\n'
 '   - Monitor for signs of shock, such as pale skin, rapid heartbeat, or shallow breathing.\n'
 '   - Do not give the person anything to eat or drink if they are unconscious or have difficulty swallowing.\n'
 '2. Treatment steps:\n'
 '   - Call emergency medical services if the fracture is severe or if the person is unable to walk or bear weight on '
 

In [167]:
paramList

[{'max_tokens': 1000, 'temp': 0, 'top_p': 0.95, 'top_k': 50},
 {'max_tokens': 1000, 'temp': 1, 'top_p': 0.85, 'top_k': 100},
 {'max_tokens': 1000, 'temp': 2, 'top_p': 0.75, 'top_k': 250},
 {'max_tokens': 1000, 'temp': 3, 'top_p': 0.65, 'top_k': 500},
 {'max_tokens': 1000, 'temp': 4, 'top_p': 0.45, 'top_k': 500}]

*   As parameters are changed over five iterations, the response gets more wordy while conveying much of the same information.  It also provides more details about what should be done post medical care, such as avoiding certain activities and getting rehabilitation care.  
*   The third of five parameter sets provides a good balance between wordiness and the amount of information provided.
* Providing context makes the queries run longer, on the order 20-30 seconds each.



## Data Preparation for RAG

### Loading the Data

In [27]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
#Load the data file provided
pdf_path = "/content/drive/MyDrive/AI_ML Course/project05/medical_diagnosis_manual.pdf"
pdf_loader = PyMuPDFLoader(pdf_path)
manual = pdf_loader.load()

### Data Overview

#### Checking the first 5 pages

In [29]:
#Verify loading of pdf file
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(manual[i].page_content,end="\n")

Page Number : 1
apgt60@hotmail.com
ICXHW2DZL4
This file is meant for personal use by apgt60@hotmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 2
apgt60@hotmail.com
ICXHW2DZL4
This file is meant for personal use by apgt60@hotmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ...........................................................................................................................................

#### Checking the number of pages

In [30]:
#Check length of loaded manual
len(manual)

4114

### Data Chunking

In [31]:
#Split the data using a text splitter with necessary attributes
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap= 50
)

document_chunks = pdf_loader.load_and_split(text_splitter)

In [32]:
#Verify length of chunks
len(document_chunks)

8659

In [33]:
#Verify content of chunks
document_chunks[0].page_content

'apgt60@hotmail.com\nICXHW2DZL4\nThis file is meant for personal use by apgt60@hotmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [34]:
#Verify content of chunks
document_chunks[-2].page_content

'Y\nYaws 1266-1267\nforest 1379\nY chromosome 3373 (see also Genetic)\nabnormalities of 3005\nYeast infection (see also Fungal infection)\nvaginal 2542, 2544, 2545\nYellow fever 1400, 1429, 1437\nhepatic inflammation in 248\nvaccine against 1172, 1437, 3441\nYellow nail syndrome 732, 1995\npleural effusion in 1997\nYellow skin (see Jaundice)\nYersinia infection 1167, 1256-1257\nY. enterocolitica infection 147\nY. pestis infection 1924\nYew poisoning 3338\nYips 1762\nYo, antibodies to 1056\nYolk sac tumor 2476\nThe Merck Manual of Diagnosis & Therapy, 19th Edition\nY\n4103\napgt60@hotmail.com\nICXHW2DZL4\nThis file is meant for personal use by apgt60@hotmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

### Embedding

In [35]:
#Pick an embedding model
embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')


In [172]:
#Create two embeddings and verify the lengths
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  1024


True

### Vector Database

In [38]:
out_dir = 'merck_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [39]:
#Load the vector db - long running process!
vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=out_dir
)

In [173]:
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

### Retriever

In [174]:
#Function to get a retreiver with search method and k value
def get_retreiver(search_type='similarity',k=2):
    return vectorstore.as_retriever(
      search_type=search_type,
      search_kwargs={'k': k}
)

In [175]:
#Run a query against the vector db
rel_docs = get_retreiver().get_relevant_documents(q1)
rel_docs

[Document(metadata={'page': 2455, 'moddate': '2026-03-06T23:36:09+00:00', 'format': 'PDF 1.7', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'trapped': '', 'creator': 'Atop CHM to PDF Converter', 'keywords': '', 'source': '/content/drive/MyDrive/AI_ML Course/project05/medical_diagnosis_manual.pdf', 'creationDate': 'D:20120615054440Z', 'total_pages': 4114, 'subject': '', 'file_path': '/content/drive/MyDrive/AI_ML Course/project05/medical_diagnosis_manual.pdf', 'creationdate': '2012-06-15T05:44:40+00:00', 'modDate': 'D:20260306233609Z', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'author': ''}, page_content='• Normalization of blood glucose levels\n• Replacement-dose corticosteroids\nPatients with septic shock should be treated in an ICU. The following should be monitored frequently (see\nalso p. 2244): systemic pressure; CVP, PAOP, or both; pulse oximetry; ABGs; blood glucose, lactate, and\nelectrolyte levels; renal function, and possibly sublingual P

In [178]:
#Check how many docs were returned
len(rel_docs)

2

### System and User Prompt Template

In [119]:
#1. A system message describing the expert's role.
#2. A user message template including context and the question.

system_message = """
You are an expert whose work is to review the manual and provide the appropriate answers from the context.
User input will have the context required by you to answer user questions.
This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.
The context ends with the token: ###EndContext.

User questions will begin with the token: ###Question.

Please answer only using the context provided in the input. Do not mention anything about the context in your final answer.

If the answer is not found in the context, respond "I don't know".

"""


qna_user_message_template = """
###Context
Here are some documents that are relevant to the question mentioned below.
{context}
###EndContext

###Question
{question}
"""

### Response Function

In [181]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = get_retreiver(k=k).get_relevant_documents(query=user_input)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = system_message + '\n' + user_message

    output( ">>>>>>>>>Calling llm with mt=" + str(max_tokens) + " temp=" + str(temperature) + " tp=" + str(top_p) + " tk=" + str(top_k) )
    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [182]:
for params in paramList:
  output(generate_rag_response(q1,max_tokens=params["max_tokens"],temperature=params["temp"],top_p=params["top_p"],top_k=params["top_k"]))
  output("                                      ---------------------------")

'>>>>>>>>>Calling llm with mt=1000 temp=0 tp=0.95 tk=50'


Llama.generate: prefix-match hit


('The context provides information on the management of sepsis and septic shock in a critical care unit. According to '
 'the text, patients with sepsis should be treated in an ICU and monitored frequently for various physiologic '
 'parameters such as systemic pressure, CVP or PAOP, pulse oximetry, ABGs, blood glucose, lactate, electrolyte levels, '
 'renal function, and sublingual PCO2. Fluid resuscitation with 0.9% saline should be given until CVP reaches 8 mm Hg '
 '(10 cm H2O) or PAOP reaches 12 to 15 mm Hg. Oliguria with hypotension is not a contraindication to vigorous fluid '
 'resuscitation. If a patient remains hypotensive after CVP or PAOP has been raised to target levels, dopamine may be '
 'given to increase mean BP to at least 60 mm Hg. If dopamine dose exceeds 20 μg/kg/min, another vasopressor such as '
 'norepinephrine may be added. However, the risks of organ hypoperfusion and acidosis from vasoconstriction caused by '
 'higher doses of dopamine and norepinephrine shou

Llama.generate: prefix-match hit


('The context provides detailed information about the management of sepsis in a critical care unit. According to the '
 'text, patients with septic shock should be treated in an ICU and monitored frequently for various physiologic '
 'parameters such as systemic pressure, CVP or PAOP, pulse oximetry, arterial blood gases, blood glucose levels, '
 'electrolyte levels, renal function, and possibly sublingual PCO2. Fluid resuscitation with 0.9% saline should be '
 'given to maintain optimal fluid levels. If a patient remains hypotensive despite reaching target CVP or PAOP levels, '
 'dopamine may be administered to increase mean BP to at least 60 mm Hg, with the possibility of adding another '
 'vasopressor if necessary. Oxygen therapy is also provided through masks or nasal prongs, and tracheal intubation and '
 'mechanical ventilation may be needed for respiratory failure. The text advises that frequent blood tests should be '
 'performed to help detect problems early, including daily e

...

('Based on the context provided, the following are the key steps for managing sepsis in a critical care unit:\n'
 '1. The patient should be treated in an ICU and monitored frequently for systemic pressure, CVP or PAOP, pulse '
 'oximetry, ABGs, blood glucose, lactate, electrolyte levels, renal function, urine output, and possibly sublingual '
 'PCO2.\n'
 '2. Fluid resuscitation with 0.9% saline should be given until CVP reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to '
 '15 mm Hg. Oliguria with hypotension is not a contraindication to vigorous fluid resuscitation, and the quantity of '
 'fluid required may far exceed normal blood volume.\n'
 '3. Dopamine may be given to increase mean BP to at least 60 mm Hg if the patient remains hypotensive after CVP or '
 'PAOP has been raised to target levels. If dopamine dose exceeds 20 μg/kg/min, another vasopressor (typically '
 'norepinephrine) may be added.\n'
 '4. Oxygen should be given by mask or nasal prongs and tracheal intubation and mec

Llama.generate: prefix-match hit


('Based on the context provided, the protocol for managing sepsis in a critical care unit includes the following '
 'steps:\n'
 '1. The patient should be treated in an ICU with experienced personnel.\n'
 '2. Monitoring should include systemic pressure, CVP or PAOP, pulse oximetry, ABGs, blood glucose, lactate and '
 'electrolyte levels, renal function, urine output, and possibly sublingual PCO2.\n'
 '3. Fluid resuscitation with 0.9% saline should be given until CVP reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to '
 '15 mm Hg. Oliguria with hypotension is not a contraindication to vigorous fluid resuscitation.\n'
 '4. Dopamine may be given to increase mean BP to at least 60 mm Hg if the patient remains hypotensive after CVP or '
 'PAOP has been raised to target levels. Norepinephrine may also be added if dopamine dose exceeds 20 μg/kg/min, but '
 'vasoconstriction poses risks of organ hypoperfusion and acidosis.\n'
 '5. Oxygen should be given by mask or nasal prongs, and tracheal intu

Llama.generate: prefix-match hit


('The context mentions that patients with sepsis should be treated in an ICU and that their blood pressure, central '
 'venous pressure (CVP), pulmonary artery occlusive pressure (PAOP), pulse oximetry, arterial blood gases (ABGs), '
 'blood glucose, lactate, electrolyte levels, renal function, and sublingual PCO2 should be monitored frequently. '
 'Fluid resuscitation with 0.9% saline is given until CVP reaches 8 mm Hg or PAOP reaches 12 to 15 mm Hg. Oliguria '
 'with hypotension is not a contraindication to vigorous fluid resuscitation, and the quantity of fluid required often '
 'exceeds the normal blood volume and may reach 10 L over 4 to 12 hours. Dopamine may be given to increase mean BP to '
 'at least 60 mm Hg if a patient remains hypotensive after CVP or PAOP has been raised to target levels. '
 'Norepinephrine may be added if dopamine dose exceeds 20 μg/kg/min, but vasoconstriction caused by higher doses of '
 'dopamine and norepinephrine poses risks of organ hypoperfusion an

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [183]:
for params in paramList:
  output(generate_rag_response(q2,max_tokens=params["max_tokens"],temperature=params["temp"],top_p=params["top_p"],top_k=params["top_k"]))
  output("                                      ---------------------------")

'>>>>>>>>>Calling llm with mt=1000 temp=0 tp=0.95 tk=50'


Llama.generate: prefix-match hit


('The common symptoms for appendicitis include abdominal pain, anorexia, and abdominal tenderness. The treatment for '
 'appendicitis is surgical removal of the appendix. Antibiotics can improve survival rate if surgery is impossible but '
 'are not curative.')
'                                      ---------------------------'
'>>>>>>>>>Calling llm with mt=1000 temp=1 tp=0.85 tk=100'


Llama.generate: prefix-match hit


('Appendicitis is a condition characterized by acute inflammation of the vermiform appendix. The typical symptoms '
 'include abdominal pain, anorexia, and abdominal tenderness. Diagnosis is clinical, often supplemented by CT or '
 'ultrasound. Treatment is surgical removal, with antibiotics administered before the procedure. In cases where '
 'surgery is impossible, antibiotics can improve the survival rate but are not curative. The preferred surgical '
 'procedures for appendicitis are open or laparoscopic appendectomy. If perforation has occurred, continuation of '
 'antibiotics until temperature and WBC count have normalized is recommended, or for a fixed course according to the '
 "surgeon's preference.")
'                                      ---------------------------'
'>>>>>>>>>Calling llm with mt=1000 temp=2 tp=0.75 tk=250'


...

('The common symptoms for appendicitis include abdominal pain, anorexia, and abdominal tenderness. According to the '
 'context, treatment for appendicitis is surgical removal. There is no mention of a cure via medicine in the provided '
 'context.')
'                                      ---------------------------'
'>>>>>>>>>Calling llm with mt=1000 temp=3 tp=0.65 tk=500'


Llama.generate: prefix-match hit


('Appendicitis is a condition characterized by inflammation of the vermiform appendix. The common symptoms include '
 'abdominal pain, anorexia, and abdominal tenderness. These symptoms can indicate the need for surgery as gangrene and '
 'perforation of the gut can occur within 6 hours from onset of symptoms in certain conditions. Treatment is surgical '
 'removal, specifically appendectomy. Antibiotics are administered before surgery to prevent infection. If surgery is '
 'impossible, antibiotics can improve survival rate but they are not curative.')
'                                      ---------------------------'
'>>>>>>>>>Calling llm with mt=1000 temp=4 tp=0.45 tk=500'


Llama.generate: prefix-match hit


('The common symptoms for appendicitis include abdominal pain, anorexia, and abdominal tenderness. The document '
 'suggests that treatment for appendicitis is surgical removal. There is no mention of curing it via medicine in the '
 'context.')
'                                      ---------------------------'


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [184]:
for params in paramList:
  output(generate_rag_response(q3,max_tokens=params["max_tokens"],temperature=params["temp"],top_p=params["top_p"],top_k=params["top_k"]))
  output("                                      ---------------------------")

'>>>>>>>>>Calling llm with mt=1000 temp=0 tp=0.95 tk=50'


Llama.generate: prefix-match hit


('Based on the context provided, alopecia areata is a condition characterized by sudden patchy hair loss. The '
 'treatment options for this condition include topical corticosteroids, oral antimalarials, retinoids, or '
 "immunosuppressants. Daily hair counts can also be done to quantify hair loss when the pull test is negative. It's "
 "important to note that treatment may need to be continued indefinitely as once it's stopped, hair loss resumes. The "
 'possible causes of alopecia areata include genetics and unclear environmental triggers.')
'                                      ---------------------------'
'>>>>>>>>>Calling llm with mt=1000 temp=1 tp=0.85 tk=100'


Llama.generate: prefix-match hit


('From the context provided, alopecia areata is mentioned as a cause of sudden patchy hair loss. The treatment options '
 'for alopecia areata include topical or intralesional corticosteroids, topical minoxidil, topical anthralin, topical '
 'immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). The context also '
 'mentions that scalp biopsy may be required for definitive diagnosis. Therefore, the possible causes behind sudden '
 'patchy hair loss could be an autoimmune disorder affecting genetically susceptible people exposed to unclear '
 'environmental triggers.\n'
 '\n'
 'Answer:\n'
 'The effective treatments or solutions for addressing sudden patchy hair loss include topical or intralesional '
 'corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid '
 "dibutylester), or psoralen plus ultraviolet A (PUVA). It's important to note that a scalp biopsy may be required for "
 'definitive 

...

('From the context provided, alopecia areata is a condition characterized by sudden patchy hair loss with no obvious '
 'skin or systemic disorder affecting genetically susceptible people exposed to unclear environmental triggers. The '
 'scalp and beard are most frequently affected areas. Alopecia areata can affect most or all of the body, resulting in '
 'alopecia universalis.\n'
 '\n'
 'The diagnosis for alopecia areata is confirmed through microscopic hair examination or scalp biopsy to differentiate '
 'scarring from nonscarring forms. Treatment options for alopecia areata include:\n'
 '1. Topical corticosteroids: effective for mild cases and localized bald spots, applied directly to the affected '
 'area.\n'
 '2. Intralesional corticosteroid injections: used for more severe cases, administered by a healthcare professional.\n'
 '3. Systemic corticosteroids: prescribed for extensive hair loss, taken orally for several months.\n'
 '4. Topical minoxidil: can be effective for male-pat

Llama.generate: prefix-match hit


('The context mentions that alopecia areata is a type of hair loss characterized by sudden patchy hair loss with no '
 'obvious skin or systemic disorder. The most common treatments for alopecia areata include topical corticosteroids, '
 'topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or '
 'psoralen plus ultraviolet A (PUVA). Daily hair counts can be done by the patient to quantify hair loss. Scalp biopsy '
 'is indicated when alopecia persists and diagnosis is in doubt.\n'
 '\n'
 'Therefore, the effective treatments for addressing sudden patchy hair loss could be topical corticosteroids, topical '
 'minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus '
 'ultraviolet A (PUVA). The possible causes behind sudden patchy hair loss could be alopecia areata.')
'                                      ---------------------------'
'>>>>>>>>>Calling llm with mt=1000 temp=4 tp

Llama.generate: prefix-match hit


('Based on the context provided, alopecia areata is a condition characterized by sudden patchy hair loss with no '
 'obvious skin or systemic disorder. The scalp and beard are most frequently affected areas, but any hairy area may be '
 'involved. Alopecia areata is thought to be an autoimmune disorder affecting genetically susceptible people exposed '
 'to unclear environmental triggers.\n'
 '\n'
 'The treatment options for alopecia areata include:\n'
 '1. Topical corticosteroids: These can be used for mild cases or for localized bald spots. They help reduce '
 'inflammation and promote hair regrowth.\n'
 "2. Immunosuppressants: In severe cases, oral immunosuppressants may be prescribed to suppress the immune system's "
 'attack on hair follicles.\n'
 '3. Minoxidil: This medication can be used topically to prolong the anagen growth phase and enlarge miniaturized '
 'follicles into mature terminal hairs. It is most effective for vertex alopecia in male-pattern or female-pattern '
 'hai

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [185]:
for params in paramList:
  output(generate_rag_response(q4,max_tokens=params["max_tokens"],temperature=params["temp"],top_p=params["top_p"],top_k=params["top_k"]))
  output("                                      ---------------------------")

'>>>>>>>>>Calling llm with mt=1000 temp=0 tp=0.95 tk=50'


Llama.generate: prefix-match hit


('I. Surgery to place monitors and treat intracranial pressure\n'
 'II. Maintaining adequate brain perfusion and oxygenation\n'
 'III. Prevention of complications of altered sensorium\n'
 'IV. Rehabilitation therapy\n'
 'V. Family education\n'
 '\n'
 'Answer:\n'
 'IV, I, II, V')
'                                      ---------------------------'
'>>>>>>>>>Calling llm with mt=1000 temp=1 tp=0.85 tk=100'


Llama.generate: prefix-match hit


("I don't know. However, the context mentions that the initial treatment consists of ensuring a reliable airway and "
 'maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed for patients with more '
 'severe injury to place monitors to track and treat intracranial pressure, decompress the brain if intracranial '
 'pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining '
 'adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important. '
 'Subsequently, many patients require rehabilitation.')
'                                      ---------------------------'
'>>>>>>>>>Calling llm with mt=1000 temp=2 tp=0.75 tk=250'


...

('Based on the context provided, the following treatments are recommended for a person with a traumatic brain injury '
 '(TBI):\n'
 '1. Ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure.\n'
 '2. Surgery to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure '
 'is increased, or remove intracranial hematomas.\n'
 '3. Maintaining adequate brain perfusion and oxygenation in the first few days after the injury.\n'
 '4. Preventing complications of altered sensorium.\n'
 '5. Rehabilitation for many patients subsequently.\n'
 '\n'
 'Therefore, the answer is: Ensuring a reliable airway, maintaining adequate ventilation, oxygenation, and blood '
 'pressure, surgery if necessary, preventing complications, and rehabilitation.')
'                                      ---------------------------'
'>>>>>>>>>Calling llm with mt=1000 temp=3 tp=0.65 tk=500'


Llama.generate: prefix-match hit


('The context explains that traumatic brain injury (TBI) is caused by physical damage to brain tissue which can '
 'temporarily or permanently impair brain function. The initial treatment includes ensuring a reliable airway and '
 'maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed for patients with more '
 'severe injuries to place monitors to track and treat intracranial pressure, decompress the brain if intracranial '
 'pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining '
 'adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important. '
 'Subsequently, many patients require rehabilitation.')
'                                      ---------------------------'
'>>>>>>>>>Calling llm with mt=1000 temp=4 tp=0.45 tk=500'


Llama.generate: prefix-match hit


('I. Initial treatment includes ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and '
 'blood pressure. Surgery is often needed in patients with more severe injury to place monitors to track and treat '
 'intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. '
 'In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing '
 'complications of altered sensorium are important. Subsequently, many patients require rehabilitation.\n'
 '\n'
 'II. Maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are '
 'important in the first few days after the injury. Early intervention by rehabilitation specialists is indispensable '
 'for maximal functional recovery. Such intervention includes prevention of secondary disabilities, prevention of '
 'pneumonia, and family education. Patients with severe cognitive dysf

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [186]:
for params in paramList:
  output(generate_rag_response(q5,max_tokens=params["max_tokens"],temperature=params["temp"],top_p=params["top_p"],top_k=params["top_k"]))
  output("                                      ---------------------------")

'>>>>>>>>>Calling llm with mt=1000 temp=0 tp=0.95 tk=50'


Llama.generate: prefix-match hit


('Based on the context provided, the necessary precautions for a person with a suspected fracture include sterile '
 'wound dressings, tetanus prophylaxis, and broad-spectrum antibiotics. The treatment steps involve immobilizing the '
 'injury using a splint to allow patients to apply ice and move more without contributing to compartment syndrome or '
 'causing complications such as deep venous thrombosis or urinary tract infections from prolonged bed rest. It is also '
 'important for the patient to seek medical care if an odor emanates from within the cast or if a fever develops, '
 'which may indicate infection. Good hygiene is essential during this time. Some fractures may require early '
 'mobilization to minimize contractures and muscle atrophy. The Merck Manual provides further information on various '
 'types of fractures and their treatment.')
'                                      ---------------------------'
'>>>>>>>>>Calling llm with mt=1000 temp=1 tp=0.85 tk=100'


Llama.generate: prefix-match hit


('Based on the context provided, here is the answer:\n'
 '1. The person with a suspected open fracture needs sterile wound dressings, tetanus prophylaxis, and broad-spectrum '
 'antibiotics.\n'
 '2. They should apply ice to the injured leg.\n'
 '3. Patients are advised to use a cane when moving around, shifting it to the lower step before descending with the '
 'bad leg.\n'
 '4. Hygiene is important for those with splints or casts; they should watch out for signs of infection such as an '
 'odor from within the cast or fever.\n'
 '5. Immobilization with bed rest may be required for some fractures but can lead to complications like deep venous '
 'thrombosis, urinary tract infections, stiffness, contractures, and muscle atrophy if prolonged.\n'
 '6. For rapidly healing injuries, early mobilization is recommended to minimize contractures and muscle atrophy and '
 'accelerate functional recovery.\n'
 '7. Fractures are diagnosed through clinical criteria and usually plain x-rays, and treat

...

('From the context provided, it appears that the individual with a suspected fracture should receive sterile wound '
 'dressings, tetanus prophylaxis, and broad-spectrum antibiotics. They should also be advised to seek medical care '
 'immediately if they notice an odor coming from within the cast or a fever. A splint may be used for some injuries to '
 'allow for mobility and ice application while minimizing complications like infection and muscle atrophy. If '
 'immobilization with bed rest is required, the person should be cautious of potential complications such as deep '
 'venous thrombosis and urinary tract infections. Prolonged immobilization can lead to stiffness, contractures, and '
 "muscle atrophy, particularly in the elderly. It's important for patients to follow good hygiene practices and move "
 'their cane to the lower step before descending with the injured leg.')
'                                      ---------------------------'
'>>>>>>>>>Calling llm with mt=1000 temp

Llama.generate: prefix-match hit


('Based on the context provided, here is the answer:\n'
 '\n'
 'The person with a suspected fractured leg should receive sterile wound dressings, tetanus prophylaxis, and '
 'broad-spectrum antibiotics (eg, a 2nd-generation cephalosporin plus an aminoglycoside). They should also be advised '
 'to use a cane and move it to the lower step shortly before descending with the bad leg. Immobilization using a '
 'splint is recommended for some stable injuries, including suspected but unproven fractures. Good hygiene is '
 'important, and patients should seek medical care if they notice an odor from within the cast or develop a fever, '
 'which may indicate infection. Prolonged immobilization of a joint can cause complications such as stiffness, '
 'contractures, and muscle atrophy. Some rapidly healing injuries may benefit from early mobilization to minimize '
 'these complications. The Merck Manual provides further information on various types of fractures and their '
 'treatment.')
'       

Llama.generate: prefix-match hit


('Answer:\n'
 'The person with a suspected fractured leg requires sterile wound dressings, tetanus prophylaxis, and broad-spectrum '
 'antibiotics. They should seek medical care immediately if an odor emanates from within the cast or if a fever '
 'develops, as these may indicate infection. A splint can be used to immobilize some stable injuries, including '
 'suspected but unproven fractures. Patients should apply ice and move more with a splint, which does not contribute '
 'to compartment syndrome. Prolonged immobilization of a joint can cause stiffness, contractures, and muscle atrophy, '
 'particularly in the elderly. Some rapidly healing injuries are best treated with resumption of active motion within '
 'the first few days or weeks (early mobilization). Fractures result from a single application of significant force to '
 'otherwise normal bone, and symptoms include pain, swelling, ecchymosis, crepitation, deformity, and abnormal motion. '
 'Occasional complications include fat

### Comments / Observations - RAG
*   Adding RAG makes the responses refer specifically to the Merck Manual more often, but not every time depending on the parameters.  
*   Overall, the second or third of five parameter sets provides a good balance between wordiness and the amount of relevant information provided.
*   The responses are also more focused on first aid and acute medical intervention, and less focused on what the patient should do after receiving medical care.
* Adding RAG makes the queries run even longer, but only to a small extent.



## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [123]:
groundedness_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
The answer should be derived only from the information presented in the context

Instructions:
1. First write down the steps that are needed to evaluate the answer as per the metric.
2. Give a step-by-step explanation if the answer adheres to the metric considering the question and context as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the answer using the evaluaton criteria and assign a score.
"""

In [124]:
relevance_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context.
Consider whether all and only the important aspects are contained in the answer when evaluating relevance.

Instructions:
1. First write down the steps that are needed to evaluate the context as per the metric.
2. Give a step-by-step explanation if the context adheres to the metric considering the question as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the context using the evaluaton criteria and assign a score.
"""

In [125]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [189]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = get_retreiver().get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    output( ">>>>>>>>>Calling llm with mt=" + str(max_tokens) + " temp=" + str(temperature) + " tp=" + str(top_p) + " tk=" + str(top_k) + " and k=" + str(k) )
    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

* Note - max_tokens of 2048 being used for this section for all Q1-Q5

In [200]:
#Checking groundedness and relevance for the middle parameter set and k=2, k=8
params = paramList[2]
output(params)
for k in [2,8]:
  ground, rel = generate_ground_relevance_response(q1, k=k, max_tokens=2048, temperature=params["temp"], top_p=params["top_p"], top_k=params["top_k"])
  output("Groundedness:")
  output(ground)
  output("Relevance:")
  output(rel)
  output("----------------------------------------------------")

{'max_tokens': 1000, 'temp': 2, 'top_k': 250, 'top_p': 0.75}
'>>>>>>>>>Calling llm with mt=2048 temp=2 tp=0.75 tk=250 and k=2'


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


'Groundedness:'
(' To evaluate the answer, the following steps should be taken:\n'
 '\n'
 '1. Identify the information in the context related to the protocol for managing sepsis in a critical care unit.\n'
 '2. Compare this information with the AI generated answer to determine if the answer is derived only from the '
 'context.\n'
 '\n'
 'The context provides detailed information on how sepsis should be managed in a critical care unit, including '
 'frequent monitoring of various physiologic parameters, fluid resuscitation, oxygen administration, and ICU support. '
 'The AI generated answer includes all these points, verbatim from the context. Therefore, the metric is followed '
 'completely.\n'
 '\n'
 'Rating: 5 (The metric is followed completely)')
'Relevance:'
(' Step 1: Identify the main aspects of the question which are "What is the protocol for managing sepsis in a critical '
 'care unit?" The key concepts here are "protocol" and "managing sepsis in a critical care unit."\n'
 '\n

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


'Groundedness:'
(' To evaluate the answer as per the metric, follow these steps:\n'
 '\n'
 '1. Identify the key information from the context regarding managing sepsis in a critical care unit.\n'
 '2. Check if the AI-generated answer is derived solely from this identified information and not influenced by any '
 'external knowledge or assumptions.\n'
 '\n'
 'Step-by-step explanation:\n'
 'The context provides comprehensive details on managing sepsis in a critical care unit, including aggressive fluid '
 'resuscitation, antibiotics based on suspected source and organisms, surgical excision or drainage of infected or '
 'necrotic tissues, supportive care, normalization of blood glucose levels with insulin infusions, replacement-dose '
 'corticosteroids, monitoring strategies, and various tests.\n'
 '\n'
 'The AI-generated answer explicitly states the protocol for managing sepsis in a critical care unit and includes all '
 'the elements mentioned above. Since the information provided in th

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [201]:
#Checking groundedness and relevance for the last parameter set and k=2, k=8
params = paramList[4]
output(params)
for k in [2,8]:
  ground, rel = generate_ground_relevance_response(q2, k=k, max_tokens=2048, temperature=params["temp"], top_p=params["top_p"], top_k=params["top_k"])
  output("Groundedness:")
  output(ground)
  output("Relevance:")
  output(rel)
  output("----------------------------------------------------")

{'max_tokens': 1000, 'temp': 4, 'top_k': 500, 'top_p': 0.45}
'>>>>>>>>>Calling llm with mt=2048 temp=4 tp=0.45 tk=500 and k=2'


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


'Groundedness:'
(' Steps to evaluate the answer:\n'
 '1. Identify the key information in the context related to appendicitis and its treatment.\n'
 '2. Determine if the AI generated answer contains only information derived from the context.\n'
 '3. Check if the answer mentions the symptoms of appendicitis correctly.\n'
 '4. Verify if the answer states that there is no cure for appendicitis through medication alone and that surgical '
 'removal is the standard treatment.\n'
 '\n'
 'Explanation:\n'
 'The AI generated answer adheres to the metric as it only contains information derived from the context. The symptoms '
 'mentioned in the answer (abdominal pain, anorexia, and abdominal tenderness) are directly taken from the context. '
 'Additionally, the statement that there is no cure for appendicitis through medication alone and that surgical '
 'removal is the standard treatment is also present in the context.\n'
 '\n'
 'Evaluation:\n'
 'The metric is followed completely.\n'
 '\n'
 'Rat

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


'Groundedness:'
(' Steps to evaluate the answer:\n'
 '1. Identify the question and the specific information being asked for in the question.\n'
 '2. Read through the context provided to understand the background information related to the topic of the question.\n'
 '3. Determine if the AI generated answer is derived solely from the information presented in the context.\n'
 '4. If the answer is derived solely from the context, evaluate the extent to which the metric (answer should be '
 'derived only from the information presented in the context) is followed.\n'
 '5. Assign a score based on the evaluation criteria (1-5) based on how closely the answer adheres to the metric.\n'
 '\n'
 'Evaluation:\n'
 'The AI generated answer does follow the metric as it is derived solely from the context provided. The common '
 'symptoms for appendicitis, the fact that medicine cannot cure appendicitis and surgery is required, and the '
 'description of the surgical procedure for appendicitis are all di

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [202]:
#Checking groundedness and relevance for the first parameter set and k=2, k=8
params = paramList[0]
output(params)
for k in [2,8]:
  ground, rel = generate_ground_relevance_response(q3, k=k, max_tokens=2048, temperature=params["temp"], top_p=params["top_p"], top_k=params["top_k"])
  output("Groundedness:")
  output(ground)
  output("Relevance:")
  output(rel)
  output("----------------------------------------------------")

{'max_tokens': 1000, 'temp': 0, 'top_k': 50, 'top_p': 0.95}
'>>>>>>>>>Calling llm with mt=2048 temp=0 tp=0.95 tk=50 and k=2'


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


'Groundedness:'
(' Steps to evaluate the answer:\n'
 '1. Identify if the answer mentions any information that is not present in the context.\n'
 '2. Check if the treatments and causes mentioned in the answer are derived solely from the context.\n'
 '3. Verify if the explanation for each treatment or cause is accurate based on the context.\n'
 '\n'
 'Explanation:\n'
 'The AI generated answer mentions topical minoxidil, finasteride, and various causes of sudden patchy hair loss '
 'including autoimmune disorders. All of this information is present in the context. The explanation for how these '
 'treatments work (prolonging anagen growth phase, enlarging miniaturized follicles, inhibiting 5α-reductase enzyme) '
 'is also derived from the context. Therefore, the answer adheres to the metric.\n'
 '\n'
 'Evaluation:\n'
 'The metric is followed completely.\n'
 '\n'
 'Rating:\n'
 'Based on the evaluation criteria, I would rate this answer as a 5 because it follows the metric completely.')
'Re

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


'Groundedness:'
(' Steps to evaluate the answer:\n'
 '1. Identify the main question and the specific information being asked for.\n'
 '2. Read through the context provided to understand the background information on hair loss, its causes, and '
 'potential treatments.\n'
 '3. Determine if the AI-generated answer is derived solely from the context without introducing any new or irrelevant '
 'information.\n'
 '4. Check if each treatment mentioned in the answer is supported by the context.\n'
 '\n'
 'The answer adheres to the metric as it mentions only the treatments and causes discussed in the context. The answer '
 'does not introduce any new or irrelevant information,')
'Relevance:'
(' To evaluate the context as per the metric, follow these steps:\n'
 '\n'
 '1. Identify the main aspects of the question: The question asks about effective treatments or solutions for sudden '
 'patchy hair loss and possible causes behind it.\n'
 '2. Determine if all important aspects are contained in the

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [203]:
#Checking groundedness and relevance for the second parameter set and k=3, k=5
params = paramList[1]
output(params)
for k in [3,5]:
  ground, rel = generate_ground_relevance_response(q4, k=k, max_tokens=2048, temperature=params["temp"], top_p=params["top_p"], top_k=params["top_k"])
  output("Groundedness:")
  output(ground)
  output("Relevance:")
  output(rel)
  output("----------------------------------------------------")

{'max_tokens': 1000, 'temp': 1, 'top_k': 100, 'top_p': 0.85}
'>>>>>>>>>Calling llm with mt=2048 temp=1 tp=0.85 tk=100 and k=3'


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


'Groundedness:'
(' Steps to evaluate the answer:\n'
 '1. Identify the key information in the context related to treatments for a person with a traumatic brain injury '
 '(TBI).\n'
 '2. Determine if the AI generated answer contains only information derived from the context without adding any new '
 'information or making assumptions beyond what is presented.\n'
 '3. Compare the information in the answer to the key information in the context to ensure consistency and accuracy.\n'
 '\n'
 'The AI generated answer follows the metric to a good extent as it primarily includes information directly derived '
 'from the context. However, it repeats some information present in the question and adds the phrase "Early '
 'intervention by rehabilitation specialists is essential for maximal functional recovery" without explicitly '
 'mentioning it in the context. Therefore, the metric is followed to a good extent but not completely.\n'
 '\n'
 'Rating: 3 (The metric is followed to a good extent)')
'Re

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


'Groundedness:'
(' To evaluate the answer as per the metric, the following steps need to be taken:\n'
 '1. Read the question and the context carefully.\n'
 '2. Identify the information related to treatments for traumatic brain injury from the context.\n'
 '3. Check if the answer is derived only from the information presented in the context.\n'
 '\n'
 'The answer states that "The context mentions that initial treatment for traumatic brain injury (TBI) consists of '
 'ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be '
 'needed to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is '
 'increased, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain '
 'perfusion and oxygenation and preventing complications are important. Subsequently, many patients require '
 'rehabilitation."\n'
 '\n'
 'The answer adheres to the metric as it

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [204]:
#Checking groundedness and relevance for the second parameter set and k=2, k=3
params = paramList[1]
output(params)
for k in [2,3]:
  ground, rel = generate_ground_relevance_response(q5, k=k, max_tokens=2048, temperature=params["temp"], top_p=params["top_p"], top_k=params["top_k"])
  output("Groundedness:")
  output(ground)
  output("Relevance:")
  output(rel)
  output("----------------------------------------------------")

{'max_tokens': 1000, 'temp': 1, 'top_k': 100, 'top_p': 0.85}
'>>>>>>>>>Calling llm with mt=2048 temp=1 tp=0.85 tk=100 and k=2'


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


'Groundedness:'
(' Steps to evaluate the answer:\n'
 '1. Identify the information in the context related to precautions and treatment for a fractured leg.\n'
 '2. Check if the AI generated answer includes all the necessary precautions and treatment steps identified in step '
 '1.\n'
 '3. Determine if the AI generated answer contains any additional or incorrect information not present in the '
 'context.\n'
 '\n'
 'Explanation:\n'
 'The context mentions that for a person with a fractured leg, they should receive sterile wound dressings, tetanus '
 'prophylaxis, and broad-spectrum antibiotics (eg, a 2nd-generation cephalosporin plus an aminoglycoside). The AI '
 'generated answer includes all these precautions and treatment steps. Additionally, the context also mentions that '
 'the person should use a cane during their recovery, and this information is also included in the AI generated '
 'answer.\n'
 '\n'
 'The answer does not include any additional or incorrect information not present

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


'Groundedness:'
(' Steps to evaluate the answer:\n'
 '1. Identify the information in the context related to precautions and treatment steps for a person with a fractured '
 'leg.\n'
 '2. Check if the AI generated answer includes only the information derived from the context provided.\n'
 '3. Evaluate the extent to which the metric is followed.\n'
 '\n'
 'Step-by-step explanation:\n'
 'The AI generated answer includes the following points:\n'
 '1. Receiving sterile wound dressings, tetanus prophylaxis, and broad-spectrum antibiotics for suspected open '
 'fractures.\n'
 '2. Seeking medical care if an odor emanates from within the cast or if a fever develops.\n'
 '3. Using a splint to immobilize the injury and allowing the patient to apply ice and move without contributing to '
 'compartment syndrome.\n'
 '4. The potential complications of prolonged immobilization (stiffness, contractures, muscle atrophy) and the '
 'benefits of early mobilization.\n'
 '5. Diagnosis of fractures through 

## Actionable Insights and Business Recommendations

- Higher values of retreival parameter "k" slightly degrades the groundedness and relevance of LLM responses using RAG.
- Higher values of max_tokens are needed for quality responses when providing context and espacially with RAG.  Otherwise, the responses get truncated and are not helpful.
- Even higher values of max_tokens are needed when evaluating the groundedness and relevance of LLM responses use RAG.
- Vector database creation time is a long running process that increases with PDF document size.  It would be ideal to create a set of these with different configurations up front and then evaluation which works best for a specific use case.


<font size=6 color='blue'>Power Ahead</font>
___